In [ ]:
# Download prod-state manifest from OneLake (ABFSS → local path)
# Sets local_prod_state_path for use in Parameters cell f-string commands.
prod_state_path = ""
if prod_state_path.startswith('abfss://'):
    import os, urllib.request
    # abfss://WORKSPACE_ID@onelake.dfs.fabric.microsoft.com/LAKEHOUSE_ID/...
    # → https://onelake.dfs.fabric.microsoft.com/WORKSPACE_ID/LAKEHOUSE_ID/...
    https_url = prod_state_path.replace('abfss://', 'https://onelake.dfs.fabric.microsoft.com/', 1)
    https_url = https_url.replace('@onelake.dfs.fabric.microsoft.com', '', 1)
    manifest_url = https_url.rstrip('/') + '/manifest.json'
    local_dir = '/tmp/prod-state'
    os.makedirs(local_dir, exist_ok=True)
    token = notebookutils.credentials.getToken('storage')
    req = urllib.request.Request(manifest_url, headers={'Authorization': f'Bearer {token}'})
    with urllib.request.urlopen(req) as resp:
        with open(f'{local_dir}/manifest.json', 'wb') as f:
            f.write(resp.read())
    local_prod_state_path = local_dir


In [ ]:
# Parameters — injected by CI (do not edit manually)
ci_target = "ephemeral_ci"
dep_command = ["dbt deps"]
clone_command = ["dbt deps", f"dbt clone --select state:modified+ --state {local_prod_state_path} --profiles-dir .github/profiles --target {ci_target}"]
build_command = ["dbt deps", f"dbt build --select state:modified+ --state {local_prod_state_path} --profiles-dir .github/profiles --target {ci_target}"]
unit_test_command = ["dbt deps", f"dbt test --select state:modified+ --select test_type:unit --state {local_prod_state_path} --profiles-dir .github/profiles --target {ci_target}"]
data_test_command = ["dbt deps", f"dbt test --select state:modified+ --exclude test_type:unit --store-failures --state {local_prod_state_path} --profiles-dir .github/profiles --target {ci_target}"]
repo_url = ""
repo_branch = ""
github_app_id = ""
github_installation_id = ""
github_pem_secret = ""
vault_url = ""
lakehouse_name = ""
lakehouse_id = ""
workspace_id = ""
workspace_name = ""
schema_name = "dbo"
run_mode = "interactive"
gate = "2"
ci_run_id = ""
head_sha = ""


In [ ]:
# Install dbt adapter
!pip install vd-dbt-fabricspark==1.9.15 -q


In [ ]:
from dbt.adapters.fabricspark.notebook import (
    run_dbt_job,
    DbtJobConfig,
    RepoConfig,
    ConnectionConfig,
)


In [ ]:
# Clone
config = DbtJobConfig(
    command=clone_command,
    repo=RepoConfig(
        url=repo_url,
        branch=repo_branch,
        github_app_id=github_app_id,
        github_installation_id=github_installation_id,
        github_pem_secret=github_pem_secret,
        vault_url=vault_url,
    ),
    connection=ConnectionConfig(
        lakehouse_name=lakehouse_name,
        lakehouse_id=lakehouse_id,
        workspace_id=workspace_id,
        workspace_name=workspace_name,
        schema_name=schema_name,
    ),
)
result = run_dbt_job(config)


In [ ]:
# Build
config = DbtJobConfig(
    command=build_command,
    repo=RepoConfig(
        url=repo_url,
        branch=repo_branch,
        github_app_id=github_app_id,
        github_installation_id=github_installation_id,
        github_pem_secret=github_pem_secret,
        vault_url=vault_url,
    ),
    connection=ConnectionConfig(
        lakehouse_name=lakehouse_name,
        lakehouse_id=lakehouse_id,
        workspace_id=workspace_id,
        workspace_name=workspace_name,
        schema_name=schema_name,
    ),
)
result = run_dbt_job(config)


In [ ]:
# Data Test
config = DbtJobConfig(
    command=data_test_command,
    repo=RepoConfig(
        url=repo_url,
        branch=repo_branch,
        github_app_id=github_app_id,
        github_installation_id=github_installation_id,
        github_pem_secret=github_pem_secret,
        vault_url=vault_url,
    ),
    connection=ConnectionConfig(
        lakehouse_name=lakehouse_name,
        lakehouse_id=lakehouse_id,
        workspace_id=workspace_id,
        workspace_name=workspace_name,
        schema_name=schema_name,
    ),
)
result = run_dbt_job(config)
